# Realized-volatility forecasting on EUR/USD — four models, one table

Runs all four models on `data/EURUSD-RV.csv` over the **same** train/test split and the
**same** test rows, then reports one comparison table.

|                | no news calendar | with news calendar |
|----------------|------------------|--------------------|
| **linear**     | HAR-RV           | HAR-X              |
| **nonlinear**  | ModernTCN        | FiLM-TCN           |

That 2x2 is the point of running all four. ModernTCN vs HAR-RV alone cannot say whether a
gain comes from the news or from the architecture, because FiLM-TCN changes both at once.
With HAR-X in the grid the two effects separate:

* **news effect**         — HAR-X vs HAR-RV (linear), FiLM-TCN vs ModernTCN (nonlinear)
* **architecture effect** — ModernTCN vs HAR-RV (no news), FiLM-TCN vs HAR-X (with news)

The last of those is the one that matters for the paper: **if FiLM-TCN cannot beat HAR-X,
the architecture is not earning its keep** — a linear model is extracting the whole signal.

Split, target and metrics are shared by every model:

* train `year <= 2023`, test `year >= 2024` (validation 2022–23 folded into train for the OLS models)
* target `Y_t^(h) = ln( (1/h) * sum_{k=0}^{h-1} RV_{t+k} )`
* MSE / MAE / QLIKE on the `ln(mean RV)` scale
* identical test rows at every horizon: **647 / 643 / 626** for h = 1 / 5 / 22

Runtime: roughly 20–50 min on a Colab GPU at `ITR = 5`. Set `ITR = 1` for a smoke pass.

## 1 · Setup

In [ ]:
import os, subprocess, sys

REPO   = "https://github.com/Mr0022/ProjectA.git"
BRANCH = "claude/optimistic-mccarthy-6zszk7"
DIR    = "/content/ProjectA"

if not os.path.isdir(DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO, DIR], check=True)
else:
    subprocess.run(["git", "-C", DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)

os.chdir(DIR)
sys.path.insert(0, DIR)
print("HEAD:", subprocess.run(["git", "log", "--oneline", "-1"],
                              capture_output=True, text=True).stdout.strip())

# --- guard: refuse to run stale code -----------------------------------------
# The aggregated target must be log(mean RV) == logsumexp(ln_RV) - log(h).
import inspect
from utils.rv import forward_log_mean
assert "rolling(h).mean()" in inspect.getsource(forward_log_mean), (
    "utils/rv.py is stale: forward_log_mean must aggregate with mean, not sum.")

# HAR_X_run.py must exist and must inherit the split from HAR_RV_run rather than
# redefining it -- that inheritance is what makes "same split" structural.
assert os.path.exists("HAR_X_run.py"), "HAR_X_run.py missing: pull the branch again."
_src = open("HAR_X_run.py").read()
assert "from HAR_RV_run import" in _src and "split_by_year" in _src, (
    "HAR_X_run.py no longer inherits the split from HAR_RV_run.")
print("target convention OK: ln(mean RV);  HAR-X inherits the HAR-RV split")

In [ ]:
# Colab ships torch / pandas / numpy / statsmodels / scipy / sklearn.
# This only fills gaps on a bare runtime.
import importlib.util, subprocess, sys   # .util is not implied by `import importlib`
missing = [p for p, m in [("pandas", "pandas"), ("numpy", "numpy"),
                          ("statsmodels", "statsmodels"), ("scipy", "scipy"),
                          ("scikit-learn", "sklearn"), ("matplotlib", "matplotlib")]
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
print("dependencies ready" + (f" (installed: {', '.join(missing)})" if missing else ""))

# torch is not auto-installed: it is preinstalled on Colab, and pip-installing it
# on a bare runtime would pull a multi-GB wheel. Only the two deep models need it,
# so report its absence rather than crashing the whole notebook here.
try:
    import torch
    print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
          "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
except ImportError:
    print("torch NOT available -- the deep-model sections will fail. "
          "The HAR-RV / HAR-X sections run fine without it.")

## 2 · Configuration

In [ ]:
EPOCHS    = 40      # max epochs per seed
PATIENCE  = 8       # early-stopping patience on validation loss
ITR       = 5       # seeds per horizon: 2021..2021+ITR-1. Set 1 for a quick smoke pass.
WORKERS   = 0       # DataLoader workers
BATCH     = 256
LR        = 0.0077943332090161695

HORIZONS  = [1, 5, 22]

# One hyper-parameter set, applied unchanged at every horizon. Only --pred_len moves.
HP = dict(
    seq_len      = 35,
    patch_size   = 32,
    patch_stride = 2,
    ffn_ratio    = 2,
    num_blocks   = 1,     # expanded to "1 1 1 1" (ModernTCN's backbone has 4 stages)
    large_size   = 31,
    small_size   = 5,
    dim          = 64,
    dropout      = 0.0,
    head_dropout = 0.0,
)

# --- event-model settings ----------------------------------------------------
EVENT_DIM    = 32          # width of the learned event-type embedding
EVENT_FUSION = "channel"   # 'channel': past events flow through the whole backbone as
                           # their own variable, so ModernTCN's cross-variable ConvFFN
                           # mixes value<->events at every stage. 'inject' adds the event
                           # embedding onto the stem feature map instead.
EVENT_FILE   = "events_daily_features.csv"   # built in section 3 from data/events_daily.csv

# --- HAR-X settings ----------------------------------------------------------
# Defaults match HAR_X_run.py. The persistence screen is NOT cosmetic: this calendar
# has person-named columns (evt_USD_Fed_s_Quarles_speech and friends) and those
# officials left office before the test period starts, so without the screen LASSO
# selects them as train-period drift markers and h=22 degrades. It inspects only the
# tail of TRAIN -- screening on test-period activity would be look-ahead.
HARX_CORR_THRESHOLD = 0.90   # merge releases that always land on the same day
HARX_MIN_EVENTS     = 30     # minimum training-window firings
HARX_RECENT_YEARS   = 2      # tail of TRAIN used to check a column is still live
HARX_MIN_RECENT     = 5      # minimum firings in that tail
HARX_PLACEBO        = 2      # scrambled-calendar falsification runs (0 to skip)

# --- two-stage refit -----------------------------------------------------------
# The deep models train on <= 2021 and spend 2022-23 on early stopping, while the
# HAR baselines fit on everything <= 2023. So a deep-vs-HAR gap is partly just a
# 20% difference in training rows -- and the withheld years are the ones closest
# in distribution to the 2024+ test period.
#
# --refit_on_val closes that: stage 1 picks the best epoch E on validation, then
# the model is RE-INITIALISED and refitted on train+val for E epochs with no early
# stopping. With REFIT_VARIANTS on, each deep model is run BOTH ways, so the table
# separates "the architecture is behind" from "the architecture had less data".
#
# Cost: doubles deep-model training time. Set False for the 2-variant run.
REFIT_VARIANTS = True

print(f"{len(HORIZONS)} horizons x {ITR} seed(s) x 2 deep models, up to {EPOCHS} epochs each")
print(f"event embedding: dim {EVENT_DIM}, fusion '{EVENT_FUSION}'")
print(f"deep HP: seq_len {HP['seq_len']}, dims {HP['dim']}, patch {HP['patch_size']}/"
      f"{HP['patch_stride']}, lr {LR:.6g}, batch {BATCH}")

In [ ]:
import re, subprocess, sys

_MEAN   = re.compile(r"^\s*mean\s+([-\d.eE+]+)\s+([-\d.eE+]+)\s+([-\d.eE+]+)\s+([-\d.eE+]+)\s*$", re.M)
_STD    = re.compile(r"^\s*std\s+([-\d.eE+]+)\s+([-\d.eE+]+)\s+([-\d.eE+]+)\s+([-\d.eE+]+)\s*$", re.M)
_SINGLE = re.compile(r"mse:\s*([-\d.eE+]+),\s*mae:\s*([-\d.eE+]+),"
                     r"\s*rse:\s*([-\d.eE+]+),\s*qlike:\s*([-\d.eE+]+)")
KEYS = ["mse", "mae", "rse", "qlike"]


def run_stream(cmd, show=r"^(train |val |test |Epoch:|>>>>>>> run|mse:|news events|\s*(seed|mean|std)\s|Early)"):
    """Run a command, echo only the interesting lines, return the full output."""
    pat, lines = re.compile(show), []
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        lines.append(line)
        if pat.search(line):
            print(line.rstrip())
    proc.wait()
    out = "".join(lines)
    if proc.returncode != 0:
        print("\n".join(out.splitlines()[-30:]))
        raise RuntimeError(f"command failed (exit {proc.returncode}): {' '.join(map(str, cmd))}")
    return out


def parse_metrics(out):
    """Return ({metric: mean}, {metric: std}). std is None for a single run."""
    m, s = _MEAN.search(out), _STD.search(out)
    if m:
        mean = {k: float(v) for k, v in zip(KEYS, m.groups())}
        std  = {k: float(v) for k, v in zip(KEYS, s.groups())} if s else None
        return mean, std
    hits = _SINGLE.findall(out)
    if not hits:
        raise RuntimeError("could not find metrics in the run output")
    return {k: float(v) for k, v in zip(KEYS, hits[-1])}, None


def moderntcn_cmd(h, use_events, refit=False):
    """
    The run.py command line for one horizon, with or without the event calendar,
    and with or without the two-stage refit on train+val.

    model_id is derived from h rather than written out, so the tag can never
    disagree with --pred_len. run.py appends '_refit' to the setting string for a
    refit run, so the two variants get separate checkpoint directories instead of
    silently overwriting each other.
    """
    rep = lambda v: [str(v)] * 4          # ModernTCN's backbone has 4 stages
    cmd = [sys.executable, "run.py", "--is_training", "1",
           "--model_id", f"{'EventTCN' if use_events else 'ModernTCN'}_h{h}",
           "--model", "ModernTCN",
           "--features", "S", "--enc_in", "1", "--dec_in", "1", "--c_out", "1",
           "--aggregate_mean", "--seq_len", str(HP["seq_len"]), "--pred_len", str(h),
           "--patch_size", str(HP["patch_size"]), "--patch_stride", str(HP["patch_stride"]),
           "--ffn_ratio", str(HP["ffn_ratio"]),
           "--num_blocks", *rep(HP["num_blocks"]),
           "--large_size", *rep(HP["large_size"]), "--small_size", *rep(HP["small_size"]),
           "--dims", *rep(HP["dim"]), "--dw_dims", *rep(HP["dim"]),
           "--dropout", str(HP["dropout"]), "--head_dropout", str(HP["head_dropout"]),
           "--revin", "1", "--use_multi_scale", "False",
           "--lradj", "TST", "--pct_start", "0.3",
           "--learning_rate", str(LR), "--batch_size", str(BATCH),
           "--train_epochs", str(EPOCHS), "--patience", str(PATIENCE),
           "--num_workers", str(WORKERS), "--itr", str(ITR)]
    if use_events:
        # --use_events also switches --data custom -> custom_events
        cmd += ["--use_events", "--event_data_path", EVENT_FILE,
                "--event_dim", str(EVENT_DIM), "--event_fusion", EVENT_FUSION]
    if refit:
        cmd += ["--refit_on_val"]
    return cmd


# Which deep runs to do: (table name, use_events, refit)
DEEP_VARIANTS = [("ModernTCN", False, False), ("FiLM-TCN", True, False)]
if REFIT_VARIANTS:
    DEEP_VARIANTS += [("ModernTCN+refit", False, True), ("FiLM-TCN+refit", True, True)]

print("helpers ready")
print("deep runs:", ", ".join(n for n, _, _ in DEEP_VARIANTS))
print(" ".join(moderntcn_cmd(5, use_events=True, refit=True)[1:]))

## 3 · Build the event calendar

`data/events_daily.csv` is one row per scheduled release; the models need one row per
**day**. `build_event_features.py` pivots it, and decides which releases get their own
column using the **training years only** — picking columns by full-sample frequency would
let the test period choose the feature space.

In [ ]:
out = run_stream([sys.executable, "build_event_features.py"], show=r"^\[")

In [ ]:
# Cheap guard before spending 20+ minutes on training: does the file satisfy
# everything Dataset_Custom_Events assumes? (--no-end-to-end skips the training
# stage, which we are about to do for real anyway.)
out = run_stream([sys.executable, "smoke_test_events.py",
                  "--event-path", EVENT_FILE, "--no-end-to-end"],
                 show=r"^(\s+\[FAIL\]|\d+ passed|FAILED)")

## 4 · Split sanity check

In [ ]:
print(run_stream([sys.executable, "check_splits.py"], show=r".").strip()[-1500:])

## 5 · HAR-RV  *(linear, no news)*

Corsi (2009). Deterministic OLS, so one run — no seed spread.

In [ ]:
import pandas as pd

run_stream([sys.executable, "HAR_RV_run.py"], show=r"^(  (Train|Test|Set)|\s+(MSE|MAE|QLIKE)\s)")

har = pd.read_csv("HAR-RV results/har_rv_all_metrics.csv")
har = har[har["split"] == "test"].set_index("horizon")[["MSE", "MAE", "QLIKE"]]
print("\nHAR-RV test metrics")
display(har.round(4))

## 6 · HAR-X  *(linear, with news)*

HAR + the scheduled-news calendar, LASSO-selected — Plíhal's N-HAR specification.

This is the baseline that makes the comparison identifiable. It sees **exactly** the
information FiLM-TCN sees: the known release schedule over the forecast window
`[t .. t+h-1]`, mean-pooled — the same statistic FiLM-TCN conditions on via
`event_embed(event_y).mean(dim=1)`.

`HAR_X_run.py` also fits **HAR-DOW** (HAR + day-of-week interactions), which is Plíhal's
actual benchmark. It is what stops a "news effect" from being a repackaged
"NFP lands on Friday" effect — reported as context in section 9.

In [ ]:
harx_cmd = [sys.executable, "HAR_X_run.py",
            "--event_path", f"./data/{EVENT_FILE}",
            "--corr_threshold", str(HARX_CORR_THRESHOLD),
            "--min_events", str(HARX_MIN_EVENTS),
            "--recent_years", str(HARX_RECENT_YEARS),
            "--min_recent", str(HARX_MIN_RECENT),
            "--placebo", str(HARX_PLACEBO)]
harx_out = run_stream(harx_cmd, show=r"^(  (Raw event|Dropped|Retained|Train|Test)|"
                                     r"\s+h=\d|  HAR(-X|-DOW)?\s+0\.|  SCRAMBLED|  REAL)")

_hx = pd.read_csv("HAR-X results/har_x_all_metrics.csv")
_hx = _hx[_hx["split"] == "test"]
harx   = _hx[_hx["model"] == "HAR-X"].set_index("horizon")[["MSE", "MAE", "QLIKE"]]
hardow = _hx[_hx["model"] == "HAR-DOW"].set_index("horizon")[["MSE", "MAE", "QLIKE"]]

print("\nHAR-X test metrics")
display(harx.round(4))

# The HAR row inside HAR_X_run.py must reproduce HAR_RV_run.py exactly. If it does not,
# the two scripts are no longer scoring the same rows and nothing below is comparable.
_har_check = _hx[_hx["model"] == "HAR"].set_index("horizon")[["MSE", "MAE", "QLIKE"]]
_delta = (_har_check - har).abs().max().max()
assert _delta < 1e-9, f"HAR rows disagree between the two scripts (max |diff| = {_delta:.2e})"
print(f"cross-check OK: HAR-X's internal HAR row reproduces HAR_RV_run.py "
      f"(max |diff| = {_delta:.1e})")

### Does the news effect survive a fake calendar?

The placebo re-runs HAR-X on a calendar whose **rows have been permuted**, destroying the
date alignment while leaving every column's sparsity and correlation structure intact.
A real news effect has to collapse here. If a scrambled calendar buys the same
improvement, the "news effect" was just LASSO fitting 200-odd extra columns to noise.

In [ ]:
placebo = pd.read_csv("HAR-X results/har_x_placebo.csv")
print("Scrambled-calendar falsification (dMSE vs HAR-DOW; DM p; news regressors kept)\n")
display(placebo.pivot(index="calendar", columns="horizon",
                      values=["dMSE_pct_vs_HAR_DOW", "DM_p"]).round(3))

## 7 · The deep models

`ModernTCN` (no calendar) and `FiLM-TCN` (`--use_events`: past releases enter the
backbone as their own variable via `--event_fusion channel`, and the **known future
schedule** over the horizon FiLM-conditions the head; the FiLM generator is
zero-initialised, so training starts as the unconditioned model and the event path only
activates where it reduces loss).

With `REFIT_VARIANTS = True` each is run twice — once as-is, once with
`--refit_on_val`. The refit pair is what tells you whether a HAR-vs-deep gap is really
about the architecture or just about the 20% of training rows the deep models otherwise
never see.

In [ ]:
import time

deep, deep_std = {}, {}
for name, use_ev, refit in DEEP_VARIANTS:
    deep[name], deep_std[name] = {}, {}
    for h in HORIZONS:
        print("\n" + "=" * 78)
        print(f"{name}   h = {h}   ({ITR} seed(s)"
              + (f", event_dim {EVENT_DIM}, fusion {EVENT_FUSION}" if use_ev else "")
              + (", two-stage refit on train+val" if refit else "") + ")")
        print("=" * 78)
        t0 = time.time()
        mean, std = parse_metrics(run_stream(moderntcn_cmd(h, use_ev, refit)))
        deep[name][h], deep_std[name][h] = mean, std
        print(f"--> {name} h={h}: MSE {mean['mse']:.4f}  MAE {mean['mae']:.4f}  "
              f"QLIKE {mean['qlike']:.4f}   ({time.time() - t0:.0f}s)")

# Back-compat aliases so the cells below read naturally.
mtcn, mtcn_std = deep["ModernTCN"], deep_std["ModernTCN"]
filmtcn, filmtcn_std = deep["FiLM-TCN"], deep_std["FiLM-TCN"]

## 8 · Comparison table

In [ ]:
import numpy as np, pandas as pd

DEEP   = {name: (deep[name], deep_std[name]) for name, _, _ in DEEP_VARIANTS}
LIN    = {"HAR-RV": har, "HAR-X": harx}
MODELS = ["HAR-RV", "HAR-X"] + [name for name, _, _ in DEEP_VARIANTS]

rows = []
for h in HORIZONS:
    for name, tbl in LIN.items():
        rows.append(dict(model=name, horizon=h, seeds=1,
                         MSE=tbl.loc[h, "MSE"], MAE=tbl.loc[h, "MAE"],
                         QLIKE=tbl.loc[h, "QLIKE"],
                         MSE_std=np.nan, MAE_std=np.nan, QLIKE_std=np.nan))
    for name, (mean_d, std_d) in DEEP.items():
        mean, stds = mean_d[h], std_d[h]
        rows.append(dict(model=name, horizon=h, seeds=ITR,
                         MSE=mean["mse"], MAE=mean["mae"], QLIKE=mean["qlike"],
                         MSE_std=stds["mse"] if stds else np.nan,
                         MAE_std=stds["mae"] if stds else np.nan,
                         QLIKE_std=stds["qlike"] if stds else np.nan))
long = pd.DataFrame(rows)

pd.set_option("display.width", 240)

wide = long.pivot(index="model", columns="horizon", values=["MSE", "MAE", "QLIKE"])
wide = wide[[(m, h) for h in HORIZONS for m in ("MSE", "MAE", "QLIKE")]]
wide.columns = pd.MultiIndex.from_tuples([(f"h={h}", m) for m, h in wide.columns])
wide = wide.reindex(MODELS)

print("Test-set losses on EUR/USD — lower is better")
print("target = ln(mean RV); identical test rows (647 / 643 / 626 at h = 1 / 5 / 22)")
print(f"deep models averaged over {ITR} seed(s); HAR-RV and HAR-X are deterministic\n")
display(wide.round(4))

# Same table with the best value per column in bold. Styling is HTML-only, so the
# plain copy above is what survives renderers that drop it.
try:
    display(wide.round(4).style.highlight_min(axis=0, props="font-weight:bold;"))
except Exception as e:
    print(f"(styled copy unavailable: {type(e).__name__})")

In [ ]:
# Where does each model win? (best per horizon x metric)
winners = pd.DataFrame(
    {f"h={h}": {m: wide.loc[:, (f"h={h}", m)].idxmin() for m in ("MSE", "MAE", "QLIKE")}
     for h in HORIZONS})
print("Best model per horizon x metric\n")
display(winners)

### The 2x2: separating the news effect from the architecture effect

Each cell is a % change in loss. **Negative = the second model is better.**

In [ ]:
def delta(a, b, h, m):
    """% change from model a to model b at horizon h on metric m. Negative = b better."""
    get = lambda name: (LIN[name].loc[h, m] if name in LIN else DEEP[name][0][h][m.lower()])
    return (get(b) / get(a) - 1) * 100

PAIRS = [
    ("news effect, linear",       "HAR-RV",    "HAR-X"),
    ("news effect, nonlinear",    "ModernTCN", "FiLM-TCN"),
    ("architecture, no news",     "HAR-RV",    "ModernTCN"),
    ("architecture, with news",   "HAR-X",     "FiLM-TCN"),
]
if REFIT_VARIANTS:
    PAIRS += [
        # How much of the deep models' deficit was simply less training data?
        ("data effect, no news",     "ModernTCN", "ModernTCN+refit"),
        ("data effect, with news",   "FiLM-TCN",  "FiLM-TCN+refit"),
        # The fair fight: both sides now fitted on everything <= 2023.
        ("architecture, like-for-like", "HAR-X",  "FiLM-TCN+refit"),
    ]

attr = pd.DataFrame(
    {(f"h={h}", m): {label: delta(a, b, h, m) for label, a, b in PAIRS}
     for h in HORIZONS for m in ("MSE", "QLIKE")})
attr.index = [f"{lab}  ({a} -> {b})" for lab, a, b in PAIRS]

print("% change in loss — negative = the second model wins\n")
display(attr.round(2))

# With the refit available, the like-for-like row is the honest one: HAR-X and
# FiLM-TCN+refit are then both fitted on every row <= 2023.
best_deep = "FiLM-TCN+refit" if REFIT_VARIANTS else "FiLM-TCN"
print(f"\nThe row that decides the paper: HAR-X -> {best_deep}.")
print("Positive means a LINEAR model on the same calendar and the same training")
print("window is at least as good, and the architecture is not paying for itself.\n")
for h in HORIZONS:
    d = delta("HAR-X", best_deep, h, "MSE")
    print(f"  h={h:2d}   MSE {d:+7.2f}%   -> {best_deep if d < 0 else 'HAR-X'} wins")

if REFIT_VARIANTS:
    print("\nHow much of the deep deficit was just training data?")
    for h in HORIZONS:
        gap_before = delta("HAR-X", "FiLM-TCN", h, "MSE")
        gap_after  = delta("HAR-X", "FiLM-TCN+refit", h, "MSE")
        print(f"  h={h:2d}   gap to HAR-X: {gap_before:+6.2f}% -> {gap_after:+6.2f}%   "
              f"(refit closed {gap_before - gap_after:+6.2f} pp)")

In [ ]:
# Context: HAR_X_run.py's own HAR -> HAR-DOW -> HAR-X ladder, which splits the
# linear news gain into calendar seasonality vs genuine news content.
ladder = pd.DataFrame({
    f"h={h}": {
        "HAR -> HAR-DOW  (day-of-week)": (hardow.loc[h, "MSE"] / har.loc[h, "MSE"] - 1) * 100,
        "HAR-DOW -> HAR-X  (news)":      (harx.loc[h, "MSE"] / hardow.loc[h, "MSE"] - 1) * 100,
    } for h in HORIZONS})
print("Decomposing the linear news gain, % change in MSE (negative = better)\n")
display(ladder.round(2))

# Diebold-Mariano, as computed by HAR_X_run.py: is the HAR-X gain distinguishable
# from zero at all? A headline % with p > 0.10 is not evidence of anything.
dm = pd.read_csv("HAR-X results/har_x_dm_tests.csv")
dm = dm[(dm.benchmark == "HAR-DOW") & (dm.model == "HAR-X")]
print("\nDiebold-Mariano (HLN-corrected), HAR-X vs HAR-DOW\n")
display(dm.pivot(index="loss", columns="horizon", values=["DM_stat", "p_value"]).round(4))

In [ ]:
# Seed spread matters: a gap smaller than the noise is not a result.
print("Per-run detail — std is across seeds; HAR-RV and HAR-X are deterministic\n")
display(long.set_index(["horizon", "model"]).round(4))

print("\nIs FiLM-TCN's edge over ModernTCN larger than seed noise?")
for h in HORIZONS:
    gap = mtcn[h]["qlike"] - filmtcn[h]["qlike"]          # > 0 means events won
    s_plain  = mtcn_std[h]["qlike"]    if mtcn_std[h]    else None
    s_events = filmtcn_std[h]["qlike"] if filmtcn_std[h] else None
    if s_plain is None or s_events is None:
        # ITR == 1: there is no spread to compare against, and pretending the
        # noise floor is 0 would make every gap look decisive.
        print(f"  h={h:2d}  QLIKE gap {gap:+.4f}   single seed — no spread; "
              f"set ITR > 1 before reading this as a result")
        continue
    noise = max(s_plain, s_events)
    verdict = ("clear"    if abs(gap) > 2 * noise else
               "marginal" if abs(gap) > noise     else "within noise")
    print(f"  h={h:2d}  QLIKE gap {gap:+.4f}   worst seed-std {noise:.4f}   -> {verdict}")

## 9 · Save

In [ ]:
import os

long.to_csv("comparison_models.csv", index=False)
attr.to_csv("comparison_models_attribution.csv")

md_lines = ["| Model | " + " | ".join(f"h={h} {m}" for h in HORIZONS
                                      for m in ("MSE", "MAE", "QLIKE")) + " |",
            "|" + "---|" * (1 + 3 * len(HORIZONS))]
for model in MODELS:
    cells_ = [model]
    for h in HORIZONS:
        r = long[(long.model == model) & (long.horizon == h)].iloc[0]
        cells_ += [f"{r.MSE:.4f}", f"{r.MAE:.4f}", f"{r.QLIKE:.4f}"]
    md_lines.append("| " + " | ".join(cells_) + " |")
table_md = "\n".join(md_lines)
open("comparison_models.md", "w").write(table_md + "\n")
print(table_md)

try:
    from google.colab import files
    files.download("comparison_models.csv")
    files.download("comparison_models.md")
    files.download("comparison_models_attribution.csv")
except Exception as e:
    print(f"\n(saved to {os.getcwd()}; auto-download unavailable: {type(e).__name__})")

---

**Reading the result.** The headline table ranks the four models; the 2x2 in section 9 says
*why* one wins. Three outcomes are worth distinguishing:

1. **FiLM-TCN beats HAR-X** — the architecture adds something beyond the calendar. That is
   the result the deep model is meant to deliver.
2. **FiLM-TCN ties HAR-X** — the calendar is doing the work and a linear model extracts it.
   Still publishable, and more honest than reporting only "FiLM-TCN beats HAR-RV".
3. **HAR-X beats FiLM-TCN** — the deep model is losing signal a linear model keeps. Worth
   reporting as-is; it is a real finding, not a failed experiment.

Check the seed-spread cell before calling any of these: with `ITR = 5`, a gap smaller than
the worst seed-std is not a result.

**Caveats carried by construction.** HAR-RV and HAR-X are fixed-estimated on `<= 2023` and
applied to `>= 2024`; Plíhal instead re-estimates on a rolling 1000-day window every day,
which would favour the linear news model. The fixed split is the conservative choice and
keeps the deep-model comparison like for like.